# 1. 環境準備與模組載入

載入量化交易模型所需的數據處理、機器學習（LightGBM）、評估指標以及繪圖視覺化套件。


In [1]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# 2. 實驗目標個股定義

定義納入多股聯合訓練與回測的台灣股市標的（以 0050 成分股為主）。


In [64]:
# 核心實驗個股：0050
stock_ids = ['1216', '1303', '2059', '2301', '2303', '2308', '2317', '2327', '2330', '2344',
       '2345', '2357', '2360', '2368', '2382', '2383', '2395', '2408', '2412', '2449',
       '2454', '2603', '2880', '2881', '2882', '2883', '2884', '2885', '2886', '2887',
       '2890', '2891', '2892', '3008', '3017', '3037', '3045', '3231', '3443', '3653',
       '3661', '3665', '3711', '4904', '4958', '5880', '6505', '6669', '7769', '8046']
# 備用擴展個股池（目前註解排除）
stock_ids += [
       '1101', '1102', '1301', '1326', '1402', '1476', '1503', '1504', '1513', '1519',
       '1560', '1590', '1605', '1717', '2105', '2207', '2312', '2313', '2324', '2337',
       '2347', '2353', '2354', '2356', '2371', '2376', '2377', '2379', '2385', '2404',
       '2409', '2455', '2474', '2492', '2542', '2609', '2610', '2615', '2618', '2633',
       '2645', '2801', '2812', '2834', '2912', '3005', '3023', '3034', '3036', '3044'
       ]

# 3. 交易策略與成本參數配置

設定量化策略的核心控管參數，包含停利點、信心門檻，以及貼近台股真實市場的摩擦成本（手續費折讓與證交稅）。


In [65]:
TAKE_PROFIT = 0.09         # 動態停利點：相對於買入價上漲 9%
STOP_LOSS = -0.15          # 動態停損點：相對於買入價下跌 9%
BUY_THRESHOLD = 0.70       # 模型發出買入訊號的最低信心度門檻 (70%)

# 台股交易摩擦成本計算
TAIWAN_FEE_RATE = 0.001425  # 券商法定基本手續費率
TAX_RATE = 0.003            # 證券交易稅率 (0.3%)
FEE_DISCOUNT = 0.6          # 券商手續費折讓（6折）

TOTAL_COST_RATIO = (TAIWAN_FEE_RATE * FEE_DISCOUNT * 2) + TAX_RATE
ENTRY_FEE = TAIWAN_FEE_RATE * FEE_DISCOUNT            # 買進手續費成本
EXIT_FEE = (TAIWAN_FEE_RATE * FEE_DISCOUNT) + TAX_RATE # 賣出綜合交易成本（含稅）

INITIAL_CAPITAL = 200000    # 回測初始全額本金（100 萬新台幣）

# 4. 資料管線、特徵工程與多股聯合模型訓練

1. 讀取真實歷史 CSV，建立未來 10 天滾動最大報酬作為 Y Label。
2. 進行前 30 天歷史日 K 資料平坦化（Lagging），建立高維度特徵矩陣 X。
3. 實施嚴格的橫截面時間序列切分（Time-Series Split），防範 Look-ahead Bias。
4. 使用 LightGBM 進行聯合訓練，並對測試集輸出預測信心度。


In [66]:
def load_all_stocks_with_lag(stock_list: list, lag_days: int = 60):
    all_stock_dfs = []
    for ticker in stock_list:
        file_path = f'../data/{ticker}_stock_data.csv'
        if not os.path.exists(file_path):
            print(f"警告: 找不到 {ticker} 的資料，跳過。")
            continue
        df = pd.read_csv(file_path)
        
        # --- 建立 Y (Label): 尋找未來 10 個交易日內的最大最高價 ---
        df['future_max_high'] = df['max'].shift(-10).rolling(window=10, min_periods=1).max()
        df['future_max_return'] = (df['future_max_high'] - df['close']) / df['close']
        df['label'] = (df['future_max_return'] >= TAKE_PROFIT).astype(int)
        
        # --- 建立 X (Features): 打包前 30 天的所有特徵進行平坦化 ---
        base_features = [col for col in df.columns if col not in ['stock_id','date', 'label', 'future_max_high', 'future_max_return']]
        feature_dict = {}
        for lag in range(lag_days):
            # 取得歷史第 lag 天的原始資料
            lagged_df = df[base_features].shift(lag)
            
            for col in base_features:
                if col in ['open', 'max', 'min', 'close']:
                    # 價格特徵去絕對化：轉換成相對於「當天收盤價」的漲跌幅比例
                    feature_dict[f"{col}_lag_{lag}"] = (lagged_df[col] - df['close']) / df['close']
                elif col == 'volume':
                    # 成交量特徵去絕對化：轉換成相對於「當天成交量」的倍數（加入極小值防除以 0）
                    feature_dict[f"volume_lag_{lag}"] = lagged_df[col] / (df['volume'] + 1e-8)
                else:
                    # 若有其他指標，則直接進行 Lag 處理
                    feature_dict[f"{col}_lag_{lag}"] = lagged_df[col]
                    
        X_all = pd.DataFrame(feature_dict)
        
        df_final = pd.concat([df[['date', 'label','future_max_return']], X_all], axis=1)
        df_final = df_final.dropna().reset_index(drop=True) # 去除頭尾邊界缺漏值
        
        # 標記類別型特徵（股票代碼）
        df_final['ticker'] = ticker
        df_final['ticker'] = df_final['ticker'].astype('category')
        all_stock_dfs.append(df_final)
        # print(f"成功處理股票 {ticker}，有效資料共 {len(df_final)} 筆")
        
    full_dataset = pd.concat(all_stock_dfs, axis=0, ignore_index=True)
    return full_dataset

print("開始串接所有股票資料...")
total_df = load_all_stocks_with_lag(stock_list=stock_ids, lag_days=30)
print(f"全部股票串接完成！總資料筆數: {len(total_df)}")

# --- 時間序列嚴格切分 (前 80% 訓練, 後 20% 測試) ---
unique_dates = sorted(total_df['date'].unique())
split_point = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_point]
print(f"資料時間軸切分點為: {split_date}")

train_df = total_df[total_df['date'] < split_date]
test_df = total_df[total_df['date'] >= split_date]

feature_cols = [col for col in total_df.columns if col not in ['date', 'label', 'future_max_return', 'future_max_high']]
X_train, y_train = train_df[feature_cols], train_df['label']
X_test, y_test = test_df[feature_cols], test_df['label']

X_train['ticker'] = X_train['ticker'].astype('category')
X_test['ticker'] = X_test['ticker'].astype('category')

# --- 核心 LightGBM 模型配置 ---
scale_weight = 1 # 目前設為 1，保留後續調整不平衡樣本權重的空間
base_model = lgb.LGBMClassifier(
        objective='binary',
        metric='average_precision',
        scale_pos_weight=scale_weight,
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        verbose=-1
)

print("開始訓練多股聯合模型...")
base_model.fit(X_train, y_train, categorical_feature=['ticker'])

# --- 預測與統計報告 ---
y_pred = base_model.predict(X_test)
y_proba = base_model.predict_proba(X_test)[:, 1]
print("\n=== 全市場聯合模型評估報告 ===")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

# --- 交易訊號過濾 ---
results = pd.DataFrame({
        'date': test_df['date'],
        'ticker': test_df['ticker'],
        'Actual_Label': y_test,
        'Confidence_Level': y_proba
}, index=test_df.index)

confidence_threshold = BUY_THRESHOLD 
high_confidence_signals = results[results['Confidence_Level'] >= confidence_threshold]
print(f"\n=== 全市場交易訊號篩選 ===")
if len(high_confidence_signals) > 0:
        actual_win_rate = high_confidence_signals['Actual_Label'].mean() * 100
        print(f"高信心度訊號的「真實跨股勝率」: {actual_win_rate:.2f}%")

開始串接所有股票資料...
全部股票串接完成！總資料筆數: 81187
資料時間軸切分點為: 2025-11-03
開始訓練多股聯合模型...


/var/folders/jd/741sx0vn7h98nzy_4kpsbgg40000gn/T/ipykernel_70313/3745763375.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['ticker'] = X_train['ticker'].astype('category')
/var/folders/jd/741sx0vn7h98nzy_4kpsbgg40000gn/T/ipykernel_70313/3745763375.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test['ticker'] = X_test['ticker'].astype('category')



=== 全市場聯合模型評估報告 ===
              precision    recall  f1-score   support

           0       0.67      0.96      0.79     10610
           1       0.64      0.12      0.20      5737

    accuracy                           0.67     16347
   macro avg       0.66      0.54      0.49     16347
weighted avg       0.66      0.67      0.58     16347

ROC AUC Score: 0.7184

=== 全市場交易訊號篩選 ===
高信心度訊號的「真實跨股勝率」: 72.47%


# 5. 回測系統輔助函式定義

封裝回測模擬中所需的底層函數，包括下單模組（考慮手續費、稅金）以及股數計算模組（支援整股交易限制）。


In [47]:
# 真實交易執行函數：動態更新帳戶可用資金與庫存持股狀態
def execute_trade (ticker, price, shares, date, action, current_capital, current_holdings) -> int | dict:
       if action == 'buy': 
              current_capital -= price * shares * (1 + ENTRY_FEE) # 扣除买入金額與買入手續費
              current_holdings[ticker] = (shares, date, price)
       else: 
              current_capital += price * shares * (1 - EXIT_FEE)  # 獲得賣出金額並扣除賣出手續費與證交稅
              del current_holdings[ticker]
       return current_capital, current_holdings

In [48]:
# 股數計算函數：支持台股整股（1000股為一張）或零股交易模式
def calculate_shares_to_buy (capital, price, isRoundLot=True) -> int:
       price *= (1 + ENTRY_FEE) 
       if isRoundLot:
              shares = capital // (price * 1000) * 1000 # 無條件捨去，計算可買進的整張張數
       else:
              shares = capital // price # 支援零股買進
       return shares

In [49]:
# 輔助資料串接函數：用以載入完整的原始 K 線資料提供給回測系統查價
def load_and_concat_data (file_list : list) -> pd.DataFrame:
       df_list = []
       for file in file_list:
              df = pd.read_csv(file)
              df_list.append(df)
       combined_df = pd.concat(df_list, ignore_index=True)
       return combined_df

# 6. 策略 A 實戰回測模擬（單檔強勢股輪動 + 9% 動態停利）

基於模型輸出的信心度進行每日橫向掃描：

1. 檢查目前持股是否已滿足 9% 停利條件（盤中最高價觸及）或持有已滿 14 天強制到期結算。
2. 當帳戶呈現空倉時，在有高信心訊號的股票中，挑選當天最強（`proba` 最高）的個股進行全額 All-in 買進。


In [73]:
# 載入全市場完整歷史K線作為回測查價庫
stock_files = ['../data/' + stock_id + '_stock_data.csv' for stock_id in stock_ids]
whole_df = load_and_concat_data(stock_files)

backtest_df = test_df[['date', 'ticker']].copy()
backtest_df['proba'] = y_proba
backtest_df['label'] = y_test

current_capital = INITIAL_CAPITAL
current_holdings = dict()
last_date = None

# 依照時間軸順序逐日進行模擬交易
for date in backtest_df['date'].unique():
       last_date = date
       daily_df = backtest_df[backtest_df['date'] == date]
       # 將當天發出訊號的股票依照模型信心度由高到低排序
       buy_df = daily_df[daily_df['label'] == 1].sort_values(by='proba', ascending=False)
       
       ticker_to_buy = None
       buy_signal = False
       sell_signal = False
       
       current_ticker = list(current_holdings.keys())[0] if current_holdings else None
       current_shares, current_entry_date, current_entry_price = current_holdings.get(current_ticker, (0, None, 0.0))

       # 【賣出檢查機制 1】：持股時間是否已到達 14 天上限
       if current_entry_date is not None and (pd.to_datetime(date) - pd.to_datetime(current_entry_date)).days >= 14:
              price_to_sell = whole_df[(whole_df['date']==date) & (whole_df['stock_id']== current_ticker)]['close'].values[0]
              sell_signal = True
              print(f"日期: {date} | 觸發賣出訊號，持股已滿 14 天。")
              
       # 【賣出檢查機制 2】：當天盤中最高價（max）是否已摸到 9% 停利門檻
       if current_ticker is not None and whole_df[(whole_df['date'] == date) & (whole_df['stock_id'] == current_ticker)]['max'].values[0] >= current_entry_price * (1 + TAKE_PROFIT):
              price_to_sell = current_entry_price * (1 + TAKE_PROFIT)
              sell_signal = True
              print(f"日期: {date} | 觸發賣出訊號，持股已達 {TAKE_PROFIT*100:.2f} 停利。") 
       #【賣出檢查機制 3】：當天盤中最低價（min）是否已摸到 15% 停損門檻
       if current_ticker is not None and whole_df[(whole_df['date'] == date) & (whole_df['stock_id'] == current_ticker)]['min'].values[0] <= current_entry_price * (1 + STOP_LOSS):
              price_to_sell = current_entry_price * (1 + STOP_LOSS)
              sell_signal = True
              print(f"日期: {date} | 觸發賣出訊號，持股已達 {STOP_LOSS*100:.2f} 停損。")      
       # 執行賣出平倉邏輯
       if sell_signal:
              current_capital, current_holdings = execute_trade(current_ticker, price_to_sell, current_shares, None, 'sell', current_capital, current_holdings)
              print(f"日期: {date} | 賣出股票: {current_ticker} | 賣出價格: {price_to_sell:.2f} | 賣出股數: {current_shares} | 剩餘資金: {current_capital:.2f}")    
              
       #【買入檢查機制】：當前無持股 且 當天有符合信心門檻的強勢股發動
       if len(current_holdings)==0 and len(buy_df) > 0 and buy_df.iloc[0]['proba'] > BUY_THRESHOLD:
              buy_signal = True
              
       # 執行買入開倉邏輯（All-in 當日信心度第一名的股票）
       if buy_signal:
              ticker_to_buy = int(buy_df.iloc[0]['ticker'])
              price_to_buy = whole_df[(whole_df['date'] == date) & (whole_df['stock_id'] == ticker_to_buy)]['close'].values[0]              
              shares_to_buy = calculate_shares_to_buy(current_capital, price_to_buy, isRoundLot=True)
              if shares_to_buy == 0:
                     continue
              current_capital, current_holdings = execute_trade(ticker_to_buy, price_to_buy, shares_to_buy, date, 'buy', current_capital, current_holdings)
              print(f"日期: {date} | 買入股票: {ticker_to_buy} | 買入價格: {price_to_buy:.2f} | 買入股數: {shares_to_buy} | 剩餘資金: {current_capital:.2f}")          

# --- 回測期末強制平倉（最後一天不論賺賠一律換回現金） ---
current_ticker = list(current_holdings.keys())[0] if current_holdings else None
current_shares, current_entry_date, current_entry_price = current_holdings.get(current_ticker, (0, None, 0.0))
if current_ticker is not None :
       price_to_sell = whole_df[(whole_df['date']==last_date) & (whole_df['stock_id']== current_ticker)]['close'].values[0]
       current_capital, current_holdings = execute_trade(current_ticker, price_to_sell, current_shares, None, 'sell', current_capital, current_holdings)
       print(f"日期: {last_date} | 賣出股票: {current_ticker} | 賣出價格: {price_to_sell:.2f} | 賣出股數: {current_shares} | 剩餘資金: {current_capital:.2f}")

日期: 2025-11-03 | 買入股票: 2408 | 買入價格: 137.00 | 買入股數: 1000.0 | 剩餘資金: 62882.86
日期: 2025-11-07 | 觸發賣出訊號，持股已達 9.00 停利。
日期: 2025-11-07 | 賣出股票: 2408 | 賣出價格: 149.33 | 賣出股數: 1000.0 | 剩餘資金: 211637.20
日期: 2025-11-07 | 買入股票: 2408 | 買入價格: 147.00 | 買入股數: 1000.0 | 剩餘資金: 64511.51
日期: 2025-11-10 | 觸發賣出訊號，持股已達 9.00 停利。
日期: 2025-11-10 | 賣出股票: 2408 | 賣出價格: 160.23 | 賣出股數: 1000.0 | 剩餘資金: 224123.83
日期: 2025-11-10 | 買入股票: 2408 | 買入價格: 161.50 | 買入股數: 1000.0 | 剩餘資金: 62485.74
日期: 2025-11-13 | 觸發賣出訊號，持股已達 9.00 停利。
日期: 2025-11-13 | 賣出股票: 2408 | 賣出價格: 176.04 | 賣出股數: 1000.0 | 剩餘資金: 237842.13
日期: 2025-11-13 | 買入股票: 1303 | 買入價格: 53.80 | 買入股數: 4000.0 | 剩餘資金: 22458.13
日期: 2025-11-17 | 觸發賣出訊號，持股已達 9.00 停利。
日期: 2025-11-17 | 賣出股票: 1303 | 賣出價格: 58.64 | 賣出股數: 4000.0 | 剩餘資金: 256121.87
日期: 2025-11-17 | 買入股票: 1303 | 買入價格: 56.50 | 買入股數: 4000.0 | 剩餘資金: 29928.64
日期: 2025-12-01 | 觸發賣出訊號，持股已滿 14 天。
日期: 2025-12-01 | 觸發賣出訊號，持股已達 9.00 停利。
日期: 2025-12-01 | 賣出股票: 1303 | 賣出價格: 61.59 | 賣出股數: 4000.0 | 剩餘資金: 275319.00
日期: 2025-12-24 | 買入股票: 2